# 01 — Dataset Exploratory Data Analysis

**Purpose:** Explore BBQ, TruthfulQA, CrowS-Pairs, and WinoBias datasets.

Outputs a clean combined dataset to `data/combined_bias_dataset.json`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
import json
from collections import Counter

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries loaded successfully.')

## 1. Load Datasets

In [ ]:
# --- BBQ (Bias Benchmark for QA) ---
# Clone first: git clone https://github.com/nyu-mll/bbq.git ../data/bbq
# Then load the JSON files
import os, glob

bbq_files = glob.glob('../data/bbq/data/*.jsonl')
bbq_data = []
for f in bbq_files:
    with open(f, 'r') as fp:
        for line in fp:
            bbq_data.append(json.loads(line))

bbq_df = pd.DataFrame(bbq_data)
print(f'BBQ: {len(bbq_df)} examples')
bbq_df.head()

In [ ]:
# --- TruthfulQA ---
# Clone first: git clone https://github.com/sylinrl/TruthfulQA.git ../data/TruthfulQA
truthful_df = pd.read_csv('../data/TruthfulQA/TruthfulQA.csv')
print(f'TruthfulQA: {len(truthful_df)} questions')
truthful_df.head()

In [ ]:
# --- CrowS-Pairs ---
# Clone: git clone https://github.com/nyu-mll/crows-pairs.git ../data/crows-pairs
crows_df = pd.read_csv('../data/crows-pairs/data/crows_pairs_anonymized.csv')
print(f'CrowS-Pairs: {len(crows_df)} pairs')
crows_df.head()

## 2. Class Distribution Analysis

In [ ]:
# Check BBQ bias categories
if 'category' in bbq_df.columns:
    category_counts = bbq_df['category'].value_counts()
    print('BBQ Bias Categories:')
    print(category_counts)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    category_counts.plot(kind='barh', ax=ax, color='#7C3AED', alpha=0.8)
    ax.set_title('BBQ Dataset — Bias Category Distribution', fontsize=14)
    ax.set_xlabel('Count')
    plt.tight_layout()
    plt.show()

In [ ]:
# CrowS-Pairs bias type distribution
if 'bias_type' in crows_df.columns:
    crows_counts = crows_df['bias_type'].value_counts()
    print('CrowS-Pairs Bias Types:')
    print(crows_counts)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    crows_counts.plot(kind='bar', ax=ax, color='#06B6D4', alpha=0.8)
    ax.set_title('CrowS-Pairs — Bias Type Distribution', fontsize=14)
    ax.set_ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 3. Sentence Length Distributions

In [ ]:
# Token length analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if 'question' in bbq_df.columns:
    bbq_df['token_count'] = bbq_df['question'].str.split().str.len()
    axes[0].hist(bbq_df['token_count'], bins=30, color='#7C3AED', alpha=0.7)
    axes[0].set_title('BBQ Question Lengths')
    axes[0].set_xlabel('Token Count')

if 'Question' in truthful_df.columns:
    truthful_df['token_count'] = truthful_df['Question'].str.split().str.len()
    axes[1].hist(truthful_df['token_count'], bins=30, color='#06B6D4', alpha=0.7)
    axes[1].set_title('TruthfulQA Question Lengths')
    axes[1].set_xlabel('Token Count')

if 'sent_more' in crows_df.columns:
    crows_df['token_count'] = crows_df['sent_more'].str.split().str.len()
    axes[2].hist(crows_df['token_count'], bins=30, color='#F59E0B', alpha=0.7)
    axes[2].set_title('CrowS-Pairs Sentence Lengths')
    axes[2].set_xlabel('Token Count')

plt.tight_layout()
plt.show()

## 4. Sample Inspection

In [ ]:
# Sample 10 examples from each dataset
print('=== BBQ Samples ===')
if len(bbq_df) > 0:
    for _, row in bbq_df.sample(min(10, len(bbq_df))).iterrows():
        print(f"  Q: {row.get('question', 'N/A')[:100]}")
        print(f"  Category: {row.get('category', 'N/A')}")
        print()

print('=== TruthfulQA Samples ===')
for _, row in truthful_df.sample(min(10, len(truthful_df))).iterrows():
    print(f"  Q: {row.get('Question', 'N/A')[:100]}")
    print(f"  Category: {row.get('Category', 'N/A')}")
    print()

## 5. Compute Basic Statistics

In [ ]:
print('=== Dataset Summary ===')
print(f'BBQ:         {len(bbq_df):>8} examples')
print(f'TruthfulQA:  {len(truthful_df):>8} questions')
print(f'CrowS-Pairs: {len(crows_df):>8} pairs')
print(f'Total:       {len(bbq_df) + len(truthful_df) + len(crows_df):>8} examples')
print()
print('Note: Class imbalance may require oversampling for minority bias types.')
print('Consider SMOTE or weighted loss during classifier training.')

## 6. Save Combined Dataset

In [ ]:
# Combine and save
os.makedirs('../data', exist_ok=True)

combined = {
    'bbq_count': len(bbq_df),
    'truthfulqa_count': len(truthful_df),
    'crows_pairs_count': len(crows_df),
    'bbq_categories': bbq_df['category'].value_counts().to_dict() if 'category' in bbq_df.columns else {},
    'crows_bias_types': crows_df['bias_type'].value_counts().to_dict() if 'bias_type' in crows_df.columns else {},
}

with open('../data/dataset_summary.json', 'w') as f:
    json.dump(combined, f, indent=2)

print('Dataset summary saved to data/dataset_summary.json')
print('\nNext: Run notebook 02 to generate the sycophancy dataset.')